In [10]:
!pip install geemap earthengine-api ipyleaflet ipywidgets -q


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [11]:
import ee
import geemap
import ipywidgets as widgets

ee.Authenticate()
ee.Initialize()

# bounds = [-4.04, 51.54, -3.32, 51.75] wales test
# bounds = [-4.95, 57.18, -4.86, 57.27] # Glenn Africk - not valid due to 2016 clouds
# bounds = [109.35, 36.45, 109.44, 36.54] # Loess plateau
bounds = [24.4, 0.4, 24.49, 0.49] # congo
# bounds = [-6.65, 5.72, -6.56, 5.81] # cote d iovoire cocoa

coord1 = (bounds[1] + bounds[3]) / 2 
coord2 = (bounds[0] + bounds[2]) / 2

Map = geemap.Map(center=[coord1, coord2], zoom=10)

In [12]:
TILE_PIXELS = 128
SCALE = 10
TILE_METERS = TILE_PIXELS * SCALE

def build_aoi(bounds):
    raw = ee.Geometry.Rectangle(bounds)
    coords = ee.List(raw.coordinates().get(0))

    raw_min_lon = ee.Number(ee.List(coords.get(0)).get(0))
    raw_min_lat = ee.Number(ee.List(coords.get(0)).get(1))
    raw_max_lon = ee.Number(ee.List(coords.get(2)).get(0))
    raw_max_lat = ee.Number(ee.List(coords.get(2)).get(1))

    tile_deg_lat = ee.Number(TILE_METERS).divide(111320)
    center_lat = raw_min_lat.add(raw_max_lat).divide(2)
    lat_cos = center_lat.multiply(3.141592653589793 / 180).cos()
    tile_deg_lon = ee.Number(TILE_METERS).divide(111320).divide(lat_cos)

    min_lon = raw_min_lon.divide(tile_deg_lon).floor().multiply(tile_deg_lon)
    min_lat = raw_min_lat.divide(tile_deg_lat).floor().multiply(tile_deg_lat)
    max_lon = raw_max_lon.divide(tile_deg_lon).ceil().multiply(tile_deg_lon)
    max_lat = raw_max_lat.divide(tile_deg_lat).ceil().multiply(tile_deg_lat)

    aoi = ee.Geometry.Rectangle([min_lon, min_lat, max_lon, max_lat])

    return aoi, min_lon, min_lat, max_lon, max_lat, tile_deg_lon, tile_deg_lat


aoi, min_lon, min_lat, max_lon, max_lat, tile_deg_lon, tile_deg_lat = build_aoi(bounds)

In [13]:
def build_gain_layer(aoi):
    worldcover = ee.Image("ESA/WorldCover/v100/2020").clip(aoi)
    is_forest = worldcover.eq(10)

    m15 = ee.Image("projects/glad/GLCLU2020/v2/LCLUC_2015").clip(aoi)
    m20 = ee.Image("projects/glad/GLCLU2020/v2/LCLUC_2020").clip(aoi)

    tree_classes = ee.List.sequence(25, 96).cat(ee.List.sequence(125, 196))
    ones = ee.List.repeat(1, tree_classes.length())

    tree2015 = m15.remap(tree_classes, ones, 0)
    tree2020 = m20.remap(tree_classes, ones, 0)

    forest_gain = tree2020.And(tree2015.Not())
    clean_gain = forest_gain.updateMask(forest_gain).focal_max(1).focal_min(1)

    gain_validated = clean_gain.And(is_forest)
    gain_binary = gain_validated.unmask(0).rename("gain")

    return gain_validated, gain_binary, is_forest


gain_validated, gain_binary, is_forest = build_gain_layer(aoi)

In [14]:
def build_grid(min_lon, min_lat, max_lon, max_lat, dx, dy):

    min_lon = ee.Number(min_lon)
    min_lat = ee.Number(min_lat)
    max_lon = ee.Number(max_lon)
    max_lat = ee.Number(max_lat)
    dx = ee.Number(dx)
    dy = ee.Number(dy)

    n_lon = max_lon.subtract(min_lon).divide(dx).floor()
    n_lat = max_lat.subtract(min_lat).divide(dy).floor()

    lon_vals = ee.List.sequence(0, n_lon.subtract(1)).map(
        lambda i: min_lon.add(ee.Number(i).multiply(dx))
    )

    lat_vals = ee.List.sequence(0, n_lat.subtract(1)).map(
        lambda j: min_lat.add(ee.Number(j).multiply(dy))
    )

    def make_row(lon):
        lon = ee.Number(lon)

        def make_cell(lat):
            lat = ee.Number(lat)

            geom = ee.Geometry.Rectangle(
                [lon, lat, lon.add(dx), lat.add(dy)],
                geodesic=False,
            )

            return ee.Feature(geom, {
                "tile_id": ee.String("tile_")
                    .cat(lon.format())
                    .cat("_")
                    .cat(lat.format())
            })

        return lat_vals.map(make_cell)

    return ee.FeatureCollection(lon_vals.map(make_row).flatten())

grid = build_grid(min_lon, min_lat, max_lon, max_lat, tile_deg_lon, tile_deg_lat)

In [15]:
def build_valid_tiles(gain_binary, full_valid, grid):

    tile_area_pixels = (
        ee.Number(TILE_METERS)
        .divide(SCALE)
        .pow(2)
    )

    gain_count = gain_binary.reduceRegions(
        collection=grid,
        reducer=ee.Reducer.sum(),
        scale=SCALE,
        tileScale=4,
    )

    valid_tiles = (
        full_valid
        .unmask(0)
        .reduceRegions(
            collection=gain_count,
            reducer=ee.Reducer.min(),
            scale=SCALE,
            tileScale=4,
        )
        .filter(
            ee.Filter.eq("min", 1)
        )
    )

    return valid_tiles, tile_area_pixels

In [16]:
def add_indices(img):
    ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    evi = img.expression(
        "2.5 * ((NIR - RED) / (NIR + 6.0 * RED - 7.5 * BLUE + 1.0))",
        {"NIR": img.select("B8"), "RED": img.select("B4"), "BLUE": img.select("B2")},
    ).rename("EVI")

    return img.select(["B2","B3","B4","B5","B6","B7","B8"]).addBands([ndvi, evi])

def mask_s2_clouds(img):
    cloud_prob = ee.Image(img.get("cloud_mask")).select("probability")

    clouds = cloud_prob.gt(40)

    scl = img.select("SCL")

    shadow = scl.eq(3)
    cirrus = scl.eq(10)

    mask = (
        clouds
        .Or(shadow)
        .Or(cirrus)
        .Not()
    )

    return img.updateMask(mask)

def get_s2_collection(geom, start, end, cloud_thresh=65):
    s2 = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterDate(start, end)
        .filterBounds(geom)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cloud_thresh))
    )

    clouds = (
        ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")
        .filterDate(start, end)
        .filterBounds(geom)
    )

    joined = ee.Join.saveFirst("cloud_mask").apply(
        primary=s2,
        secondary=clouds,
        condition=ee.Filter.equals(
            leftField="system:index",
            rightField="system:index",
        ),
    )

    return ee.ImageCollection(joined).map(mask_s2_clouds)


def s2_composite(geom, year):
    start = "2015-01-01" if year == 2016 else f"{year}-01-01"
    end = "2016-12-31" if year == 2016 else f"{year}-12-31"
    ic = (
        get_s2_collection(geom, start, end, 65)
        .map(add_indices)
    )
    reduced = ic.reduce(ee.Reducer.percentile([25]))

    fallback = ee.Image.constant(0).rename("B2_p25") \
        .addBands(ee.Image.constant(0).rename("B3_p25")) \
        .addBands(ee.Image.constant(0).rename("B4_p25")) \
        .addBands(ee.Image.constant(0).rename("B5_p25")) \
        .addBands(ee.Image.constant(0).rename("B6_p25")) \
        .addBands(ee.Image.constant(0).rename("B7_p25")) \
        .addBands(ee.Image.constant(0).rename("B8_p25")) \
        .addBands(ee.Image.constant(0).rename("NDVI_p25")) \
        .addBands(ee.Image.constant(0).rename("EVI_p25")) \
        .updateMask(ee.Image.constant(0))  # all masked = no valid pixels
    reduced = ee.Algorithms.If(ic.size().eq(0), fallback, reduced)
    reduced = ee.Image(reduced)
    return reduced.select(
        ["B2_p25","B3_p25","B4_p25","B5_p25","B6_p25","B7_p25","B8_p25","NDVI_p25","EVI_p25"],
        ["B2","B3","B4","B5","B6","B7","B8","NDVI","EVI"],
    )


def s2_peak_composite(geom, year):
    centroid = ee.Geometry(geom).centroid(maxError=1)
    lat = ee.Number(centroid.coordinates().get(1))
    north = lat.gt(0)

    if year == 2016:
        start = ee.String(ee.Algorithms.If(north, "2015-05-01", "2015-11-01"))
        end   = ee.String(ee.Algorithms.If(north, "2016-09-30", "2017-03-31"))
    else:
        start = ee.String(ee.Algorithms.If(north, f"{year}-05-01", f"{year}-11-01"))
        end   = ee.String(ee.Algorithms.If(north, f"{year}-09-30", f"{year+1}-03-31"))

    return (
        get_s2_collection(geom, start, end, 65)
        .map(add_indices)
        .select(["NDVI", "EVI"])
        .median()
    )


def s1_composite(geom, year):
    med = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterDate(f"{year}-01-01", f"{year}-12-31")
        .filterBounds(geom)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .select(["VV", "VH"])
        .median()
    )
    return med.addBands(med.select("VV").divide(med.select("VH")).rename("VVVH"))


def dw_composite(geom, year):
    return (
        ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
        .filterDate(f"{year}-01-01", f"{year}-12-31")
        .filterBounds(geom)
        .select(["trees", "crops", "built"])
        .median()
    )


s2_2016 = s2_composite(aoi, 2016)
s2_2020 = s2_composite(aoi, 2020)
s2_2025 = s2_composite(aoi, 2025)

In [17]:
def score_tile_viability(tile, gain_validated, gain_height):

    geom = tile.geometry()
    gain_mask = gain_validated.selfMask()

    s2_t0 = s2_peak_composite(geom, 2016)
    s2_t1 = s2_peak_composite(geom, 2020)

    ndvi_diff = (
        s2_t1.select("NDVI")
        .subtract(s2_t0.select("NDVI"))
        .updateMask(gain_mask)
        .rename("NDVI_diff")
    )

    stats = ndvi_diff.reduceRegion(
        reducer=ee.Reducer.median(),
        geometry=geom,
        scale=SCALE,
        maxPixels=1e13,
    )

    ndvi_delta = ee.Number(
        ee.Algorithms.If(
            stats.get("NDVI_diff"),
            stats.get("NDVI_diff"),
            0,
        )
    )

    canopy_stats = (
        gain_height
        .updateMask(gain_mask)
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geom,
            scale=SCALE,
            maxPixels=1e13,
        )
    )

    canopy_mean = ee.Number(
        ee.Algorithms.If(
            canopy_stats.get("canopy_gain_height"),
            canopy_stats.get("canopy_gain_height"),
            0,
        )
    )

    return tile.set({
        "ndvi_delta": ndvi_delta,
        "gain_canopy_mean": canopy_mean,
    })

In [ ]:
def s2_availability(geom, year):
    start = "2015-01-01" if year == 2016 else f"{year}-01-01"
    end = "2016-12-31" if year == 2016 else f"{year}-12-31"

    bands = ["B2","B3","B4","B5","B6","B7","B8"]

    ic = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterDate(start, end)
        .filterBounds(geom)
        .select(bands)
    )

    count = ic.size()

    valid = ee.Image(
        ee.Algorithms.If(
            count.gt(0),
            ic.map(lambda img: img.mask().reduce(ee.Reducer.min()))
              .reduce(ee.Reducer.max()),
            ee.Image(0)
        )
    )

    return valid.unmask(0)


full_valid = (
    s2_availability(aoi, 2016)
    .And(s2_availability(aoi, 2020))
    .And(s2_availability(aoi, 2025))
    .rename("valid")
)

valid_flag = ee.Number(
    full_valid.reduceRegion(
        reducer=ee.Reducer.max(),
        geometry=aoi,
        scale=10,
        maxPixels=1e13
    ).get("valid")
).getInfo()

print("FULL_VALID:", valid_flag)

if valid_flag != 1:
    raise SystemExit("full_valid failed — stopping notebook")


EEException: Dictionary.get: Dictionary does not contain key: 'constant'.

In [ ]:
fabdem = (
    ee.ImageCollection("projects/sat-io/open-datasets/FABDEM")
    .filterBounds(aoi)
    .mosaic()
    .clip(aoi)
)

slope = ee.Terrain.slope(fabdem)

canopy_raw = (
    ee.ImageCollection(
        "projects/sat-io/open-datasets/facebook/meta-canopy-height"
    )
    .mosaic()
)

canopy = (
    canopy_raw
    .clip(aoi)
    .rename("canopy_height")
    .updateMask(canopy_raw.gte(0))
)

gain_height = canopy.updateMask(gain_validated).rename("canopy_gain_height")

jrc = ee.Image("JRC/GFC2020_subtypes/V1").clip(aoi).rename("jrc_forest_type")

nat_forest = (
    ee.ImageCollection(
        "projects/nature-trace/assets/forest_typology/natural_forest_2020_v1_0_collection"
    )
    .mosaic()
    .select("B0")
    .divide(250)
    .clip(aoi)
    .unmask(0)
    .rename("natural_forest_prob")
)

gem = (
    ee.ImageCollection("projects/sat-io/open-datasets/GEM-Forest/GEM-Forest_2020")
    .mosaic()
    .select("b1")
    .clip(aoi)
    .unmask(0)
    .rename("agrocrop_presence")
)

gem_treecrop = gem.eq(2).selfMask()

metrics_img = ee.Image.cat([
    gain_binary.rename("gain"),
    gain_height.rename("canopy"),
    jrc.rename("jrc"),
    nat_forest.rename("nat"),
    gem.rename("agro")
])

tile_metrics = metrics_img.reduceRegions(
    collection=grid,
    reducer=ee.Reducer.mean(),
    scale=SCALE,
    tileScale=4
)

jrc_mode = jrc.reduceRegions(
    collection=grid,
    reducer=ee.Reducer.mode(),
    scale=SCALE,
    tileScale=4
)

def merge(f):
    tid = f.get("system:index")
    jm = jrc_mode.filter(ee.Filter.eq("system:index", tid)).first()
    return f.set({"jrc_mode": jm.get("mode")})


tile_metrics = tile_metrics.map(merge)

GAIN_PCT_MIN  = 0.01
NDVI_DELTA_MIN = 0.0
GAIN_CANOPY_MIN = 1.0

gain_tiles = tile_metrics.filter(ee.Filter.gte("gain", GAIN_PCT_MIN))

scored_tiles = gain_tiles.map(
    lambda t: score_tile_viability(t, gain_validated, gain_height)
)

filtered_tiles = scored_tiles.filter(
    ee.Filter.And(
        ee.Filter.gt("ndvi_delta", NDVI_DELTA_MIN),
        ee.Filter.gte("gain_canopy_mean", GAIN_CANOPY_MIN),
    )
)

tiles_with_stats = filtered_tiles.map(lambda t: t.set({
    "gain_mean": gain_binary.reduceRegion(
        ee.Reducer.mean(), t.geometry(), SCALE, maxPixels=1e13
    ).get("gain"),

    "canopy_mean": gain_height.reduceRegion(
        ee.Reducer.mean(), t.geometry(), SCALE, maxPixels=1e13
    ).get("canopy_gain_height"),

    "jrc_mode": jrc.reduceRegion(
        ee.Reducer.mode(), t.geometry(), SCALE, maxPixels=1e13
    ).get("jrc_forest_type"),

    "nat_mean": nat_forest.reduceRegion(
        ee.Reducer.mean(), t.geometry(), SCALE, maxPixels=1e13
    ).get("natural_forest_prob"),
}))

In [ ]:
import ee
import geemap
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display

DW_COLOURS = {
    0: "#419BDF",
    1: "#397D49",
    2: "#88B053",
    3: "#7A87C6",
    4: "#E49635",
    5: "#DFC35A",
    6: "#C4281B",
    7: "#A59B8F",
    8: "#B39FE1",
}

DW_CLASS_NAMES = [
    "Water",
    "Trees",
    "Grass",
    "Flooded veg",
    "Crops",
    "Shrub",
    "Built",
    "Bare",
    "Snow/ice",
]

CHART_YEARS = list(range(2015, 2026))

JRC_LABELS = {
    1: "Naturally regenerating",
    10: "Primary forest",
    20: "Planted/Plantation",
}

dw_chart_stack = ee.Image([
    (
        ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
        .filterDate(f"{yr}-01-01", f"{yr}-12-31")
        .filterBounds(aoi)
        .select("trees")
        .mean()
        .rename(f"trees_{yr}")
    )
    for yr in CHART_YEARS
])

info_output = widgets.Output(
    layout=widgets.Layout(
        width="600px",
        max_height="260px",
        overflow_y="auto",
        overflow_x="auto",
        border="1px solid #ccc",
        padding="8px",
    )
)

chart_output = widgets.Output(
    layout=widgets.Layout(
        width="750px",
        border="1px solid #ccc",
        padding="8px",
    )
)

with info_output:
    print("Click a filtered tile.")

def get_dw_chart_data(geom):

    gain_mask = gain_validated.selfMask().clip(geom)

    trees_vals = (
        dw_chart_stack
        .updateMask(gain_mask)
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geom,
            scale=10,
            maxPixels=1e13,
        )
        .getInfo()
    )

    label_pcts = {}

    for yr in CHART_YEARS:

        hist = (
            ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
            .filterDate(f"{yr}-01-01", f"{yr}-12-31")
            .filterBounds(geom)
            .select("label")
            .reduce(ee.Reducer.mode())
            .rename("label")
            .clip(geom)
            .updateMask(gain_mask)
            .reduceRegion(
                reducer=ee.Reducer.frequencyHistogram(),
                geometry=geom,
                scale=10,
                maxPixels=1e13,
            )
            .get("label")
        )

        hist = ee.Dictionary(
            ee.Algorithms.If(
                hist,
                hist,
                ee.Dictionary({})
            )
        )

        vals = ee.List(hist.values())

        total = ee.Number(
            ee.Algorithms.If(
                ee.Number(vals.reduce(ee.Reducer.sum())).gt(0),
                vals.reduce(ee.Reducer.sum()),
                1
            )
        )

        label_pcts[yr] = (
            hist.map(
                lambda k, v: ee.Number(v)
                .divide(total)
                .multiply(100)
            )
            .getInfo()
        )

    return trees_vals, label_pcts

def plot_dw_chart(tile_id, trees_vals, label_pcts):

    years = CHART_YEARS

    fig, (ax1, ax2) = plt.subplots(
        2,
        1,
        figsize=(7, 6),
        sharex=True
    )

    fig.suptitle(
        f"DW time series — gain pixels\n{tile_id}",
        fontsize=11
    )

    probs = [
        trees_vals.get(f"trees_{yr}")
        for yr in years
    ]

    valid = [
        (yr, p)
        for yr, p in zip(years, probs)
        if p is not None
    ]

    if valid:

        vx, vy = zip(*valid)

        ax1.plot(
            vx,
            vy,
            linewidth=2,
            marker="o",
            markersize=4,
            color="#397D49"
        )

        ax1.fill_between(
            vx,
            vy,
            alpha=0.15,
            color="#397D49"
        )

    ax1.set_ylabel("Trees probability")
    ax1.set_ylim(0, 1)
    ax1.grid(True, alpha=0.3)

    all_classes = sorted({
        int(k)
        for yr in years
        for k in label_pcts.get(yr, {})
    })

    bar_bottoms = [0] * len(years)

    for cls in all_classes:

        heights = [
            label_pcts.get(yr, {}).get(str(cls), 0)
            for yr in years
        ]

        ax2.bar(
            years,
            heights,
            bottom=bar_bottoms,
            width=0.7,
            color=DW_COLOURS.get(cls, "#cccccc"),
            label=DW_CLASS_NAMES[cls]
        )

        bar_bottoms = [
            b + h
            for b, h in zip(bar_bottoms, heights)
        ]

    ax2.set_ylabel("Label %")
    ax2.set_ylim(0, 100)

    ax2.set_xticks(years)

    ax2.set_xticklabels(
        [str(y) for y in years],
        rotation=45
    )

    ax2.grid(True, alpha=0.3)

    ax2.legend(
        fontsize=7,
        bbox_to_anchor=(1.02, 1),
        loc="upper left"
    )

    plt.tight_layout()

    return fig

Map_main = geemap.Map()

Map_main.centerObject(aoi, 10)

Map_main.addLayer(
    s2_2016.clip(aoi).select(["B4", "B3", "B2"]),
    {"min": 200, "max": 3000, "gamma": 1.3},
    "S2 2016"
)

Map_main.addLayer(
    s2_2020.clip(aoi).select(["B4", "B3", "B2"]),
    {"min": 200, "max": 3000, "gamma": 1.3},
    "S2 2020"
)

Map_main.addLayer(
    s2_2025.clip(aoi).select(["B4", "B3", "B2"]),
    {"min": 200, "max": 3000, "gamma": 1.3},
    "S2 2025"
)

Map_main.addLayer(
    gain_validated.selfMask(),
    {"palette": ["00E676"]},
    "Gain"
)

Map_main.addLayer(
    is_forest.selfMask(),
    {"palette": ["006400"]},
    "Forest 2020"
)

canopy_vis = {
    "min": 0,
    "max": 40,
    "palette": [
        "#f7fcf5",
        "#e5f5e0",
        "#c7e9c0",
        "#a1d99b",
        "#74c476",
        "#41ab5d",
        "#238b45",
        "#005a32"
    ]
}

Map_main.addLayer(
    gain_height,
    canopy_vis,
    "Canopy Gain Height"
)

Map_main.addLayer(
    jrc,
    {
        "min": 1,
        "max": 20,
        "palette": [
            "78c679",
            "006837",
            "cc6600",
        ],
    },
    "JRC Forest Type"
)

Map_main.addLayer(
    nat_forest,
    {
        "min": 0,
        "max": 1,
        "palette": ["white", "green"],
    },
    "Natural Forest Prob"
)

gem_overlap = gem_treecrop.And(gain_validated.selfMask())

Map_main.addLayer(
    gem_overlap,
    {"palette": ["#ff0000"]},
    "GEM ∩ Gain overlap"
)

Map_main.addLayer(
    tiles_with_stats.style(
        color="00FF00",
        fillColor="00000000",
        width=2
    ),
    {},
    "Filtered Tiles"
)

Map_main.addLayer(
    tile_metrics.style(
        color="#FFAC1C",
        fillColor="00000000",
        width=2
    ),
    {},
    "All Tiles"
)

s2_t0 = s2_peak_composite(aoi, 2016)
s2_t1 = s2_peak_composite(aoi, 2020)

ndvi_delta_img = (
    s2_t1.select("NDVI")
    .subtract(s2_t0.select("NDVI"))
    .rename("ndvi_delta")
    .clip(aoi)
)

Map_main.addLayer(
    ndvi_delta_img,
    {
        "min": -0.4,
        "max": 0.4,
        "palette": [
            "#d73027",  # red
            "#ffffff",  # white
            "#1a9850",  # green
        ],
    },
    "NDVI Δ 2016→2020"
)

Map_main.addLayerControl()

def handle_click(**kwargs):

    if kwargs.get("type") != "click":
        return

    lat = kwargs["coordinates"][0]
    lon = kwargs["coordinates"][1]

    point = ee.Geometry.Point([lon, lat])

    info = (
        tiles_with_stats
        .filterBounds(point)
        .first()
        .getInfo()
    )

    with info_output:

        info_output.clear_output()

        if not info:
            print("No filtered tile here.")
            return

        p = info["properties"]

        geom = ee.Geometry(info["geometry"])

        def fmt(v, d=2):
            return f"{float(v):.{d}f}" if v is not None else "N/A"

        jrc_raw = p.get("jrc_mode")

        rows = [
            ("Tile ID", p.get("tile_id", "N/A")),
            ("Gain area (%)", fmt(p.get("gain_mean") * 100) + "%"),
            ("Canopy height (gain area) (m)", fmt(p.get("canopy_mean"))),
            ("Natural forest (%)", fmt(p.get("nat_mean") * 100) + "%"),
            ("JRC type", JRC_LABELS.get(int(jrc_raw), "N/A") if jrc_raw else "N/A"),
        ]

        width = max(len(r[0]) for r in rows) + 2

        for k, v in rows:
            print(f"{k:<{width}}: {v}")

        print("\nFetching DW time series...")

    trees_vals, label_pcts = get_dw_chart_data(geom)

    fig = plot_dw_chart(
        p.get("tile_id", ""),
        trees_vals,
        label_pcts
    )

    with chart_output:

        chart_output.clear_output(wait=True)

        display(fig)

        plt.close(fig)

Map_main.on_interaction(handle_click)

Map_main.add_widget(
    info_output,
    position="bottomright"
)

In [ ]:
canopy_legend_dict = {
    "0-2 m (bare/grass)": "#f7fcf5",
    "2-5 m (shrubs)": "#e5f5e0",
    "5-10 m (young forest)": "#c7e9c0",
    "10-15 m": "#a1d99b",
    "15-20 m": "#74c476",
    "20-25 m": "#41ab5d",
    "25-30 m": "#238b45",
    "30+ m (tall forest)": "#005a32",
}

Map_main.add_legend(
    title="Canopy Height (m)",
    legend_dict=canopy_legend_dict
)

Map_main

Map(center=[5.766439880951106, -6.604730952373255], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
display(chart_output)

Output(layout=Layout(border_bottom='1px solid #ccc', border_left='1px solid #ccc', border_right='1px solid #cc…